# 01 — Data Preparation

In [1]:
import pandas as pd
import numpy as np
import warnings
import os

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)


## 1. Load Raw Data


In [2]:
DATA_DIR = os.path.join('..', 'D:/Github/DRM_key_demand_forecasting/Dataset')

df_iptv = pd.read_csv(
    os.path.join(DATA_DIR, 'Log_Get_DRM_List.csv'),
    parse_dates=['Date']
)
print(f'Log_Get_DRM_List  : {df_iptv.shape[0]:>10,} rows')

df_fimplus = pd.read_csv(
    os.path.join(DATA_DIR, 'Log_Fimplus_MovieID.csv'),
    parse_dates=['date']
)
print(f'Log_Fimplus_MovieID: {df_fimplus.shape[0]:>10,} rows')

df_bhd = pd.read_csv(
    os.path.join(DATA_DIR, 'Log_BHD_MovieID.csv'),
    parse_dates=['DATE']
)
print(f'Log_BHD_MovieID   : {df_bhd.shape[0]:>10,} rows')

df_movies = pd.read_csv(
    os.path.join(DATA_DIR, 'MV_PropertiesShowVN.csv')
)
print(f'MV_PropertiesShowVN: {df_movies.shape[0]:>10,} rows')

Log_Get_DRM_List  :    663,594 rows
Log_Fimplus_MovieID:  1,687,012 rows
Log_BHD_MovieID   :    935,263 rows
MV_PropertiesShowVN:     28,807 rows


In [3]:
print(' Log_Get_DRM_List (IPTV)')
display(df_iptv.head(3))
print(df_iptv.dtypes)

print('Log_Fimplus_MovieID')
display(df_fimplus.head(3))
print(df_fimplus.dtypes)

print('Log_BHD_MovieID')
display(df_bhd.head(3))
print(df_bhd.dtypes)

print('MV_PropertiesShowVN')
display(df_movies.head(3))
print(df_movies.dtypes)

 Log_Get_DRM_List (IPTV)


,CustomerID,Date,Mac
0,378250,2020-04-12,B046FCB46C3A
1,1636894,2020-04-19,DCA26622E8BD
2,1404674,2020-04-22,90324B0EE445


CustomerID             int64
Date          datetime64[ns]
Mac                   object
dtype: object
Log_Fimplus_MovieID


,CustomerID,MovieId,Ftype,realtimeplaying,date,folder,utype
0,1665166,100060921,1,63404,2019-05-04 00:23:41,4035,4
1,1949528,100069961,1,938,2019-05-04 00:23:46,4035,4
2,1685734,100067273,1,2544,2019-05-04 00:23:48,4021,4


CustomerID                  int64
MovieId                     int64
Ftype                       int64
realtimeplaying             int64
date               datetime64[ns]
folder                      int64
utype                       int64
dtype: object
Log_BHD_MovieID


,CustomerID,MovieID,FTYPE,REALTIMEPLAYING,DATE,folder,Utype
0,1177679,100059478,3,30,2019-05-01 14:54:08,3065,4
1,1021748,100059478,3,3321,2019-05-01 14:54:12,3065,4
2,1149426,100059479,3,3623,2019-05-01 14:54:48,3063,4


CustomerID                  int64
MovieID                     int64
FTYPE                       int64
REALTIMEPLAYING             int64
DATE               datetime64[ns]
folder                      int64
Utype                       int64
dtype: object
MV_PropertiesShowVN


,id,toptitle,titleEN,release,actors,directors,Producers,PublishCountry,Duration,isDRM
0,100000001,Tâm lý,Innocent Steps,2005.00,"Mun Geun-yung, Park Kun-hyung, Jung Yu-mi",Park Young-hoon,Culture Cap Media,Hàn Quốc,720.00,0
1,100000002,Hành động,AAofUltron_1080p_10M_AC3_H264_TS,2015.00,"Robert Downey Jr., Mark Ruffalo, Chris Evans",Joss Whedon,Marvel Studios,Mỹ,48.00,0
2,100000003,Hành động,AAofUltron_1080p_8M_AC3_H264_TS,2015.00,"Robert Downey Jr., Mark Ruffalo, Chris Evans",Joss Whedon,Marvel Studios,Mỹ,48.00,0


id                  int64
toptitle           object
titleEN            object
release           float64
actors             object
directors          object
Producers          object
PublishCountry     object
Duration          float64
isDRM               int64
dtype: object


In [4]:
drm_movie_ids = df_movies.loc[df_movies['isDRM'] == 1, 'id'].unique()
print(f'DRM-protected movies: {len(drm_movie_ids)}')

DRM-protected movies: 6605


In [5]:

iptv_base = df_iptv[['Date', 'CustomerID']].copy()
iptv_base.columns = ['LogDate', 'CustomerID']
iptv_base['Source'] = 'IPTV'
iptv_base['LogDate'] = pd.to_datetime(iptv_base['LogDate']).dt.normalize()

fimplus_drm = df_fimplus[df_fimplus['MovieId'].isin(drm_movie_ids)].copy()
fimplus_base = fimplus_drm[['date', 'CustomerID']].copy()
fimplus_base.columns = ['LogDate', 'CustomerID']
fimplus_base['Source'] = 'Fimplus'
fimplus_base['LogDate'] = pd.to_datetime(fimplus_base['LogDate']).dt.normalize()

bhd_drm = df_bhd[df_bhd['MovieID'].isin(drm_movie_ids)].copy()
bhd_base = bhd_drm[['DATE', 'CustomerID']].copy()
bhd_base.columns = ['LogDate', 'CustomerID']
bhd_base['Source'] = 'BHD'
bhd_base['LogDate'] = pd.to_datetime(bhd_base['LogDate']).dt.normalize()

df_drm_base = pd.concat([iptv_base, fimplus_base, bhd_base], ignore_index=True)

print(f'VW_DRM_BASE total records: {len(df_drm_base):,}')
print(f'Date range: {df_drm_base["LogDate"].min()} → {df_drm_base["LogDate"].max()}')
print(f'\nRecords by source:')
print(df_drm_base['Source'].value_counts())

VW_DRM_BASE total records: 2,733,204
Date range: 2019-05-01 00:00:00 → 2020-06-01 00:00:00

Records by source:
Source
Fimplus    1687012
IPTV        663594
BHD         382598
Name: count, dtype: int64


In [6]:
df_daily = (
    df_drm_base
    .groupby('LogDate')['CustomerID']
    .nunique()
    .reset_index()
    .rename(columns={'CustomerID': 'Total_Daily_Keys'})
    .sort_values('LogDate')
    .reset_index(drop=True)
)

print(f'Daily key count: {len(df_daily)} days')
print(f'Date range: {df_daily["LogDate"].min().date()} → {df_daily["LogDate"].max().date()}')
display(df_daily.describe())

Daily key count: 398 days
Date range: 2019-05-01 → 2020-06-01


,LogDate,Total_Daily_Keys
count,398,398.00
mean,2019-11-15 12:00:00.000000256,3094.09
min,2019-05-01 00:00:00,38.00
25%,2019-08-08 06:00:00,80.00
50%,2019-11-15 12:00:00,676.00
75%,2020-02-22 18:00:00,3942.25
max,2020-06-01 00:00:00,15478.00
std,NaN,4019.77


In [7]:
# Source-level breakdown per day (for later analysis)
df_daily_source = (
    df_drm_base
    .groupby(['LogDate', 'Source'])['CustomerID']
    .nunique()
    .reset_index()
    .rename(columns={'CustomerID': 'Keys'})
    .pivot_table(index='LogDate', columns='Source', values='Keys', fill_value=0)
    .reset_index()
)

display(df_daily_source.head())

Source,LogDate,BHD,Fimplus,IPTV
0,2019-05-01,93.00,4520.00,0.00
1,2019-05-02,68.00,3780.00,0.00
2,2019-05-03,73.00,4092.00,0.00
3,2019-05-04,89.00,4522.00,0.00
4,2019-05-05,83.00,4567.00,0.00


In [8]:
full_date_range = pd.date_range(
    start=df_daily['LogDate'].min(),
    end=df_daily['LogDate'].max(),
    freq='D'
)

missing_dates = full_date_range.difference(df_daily['LogDate'])
print(f'Missing dates in range: {len(missing_dates)}')
if len(missing_dates) > 0:
    print(missing_dates.tolist())

Missing dates in range: 0


In [9]:
df_daily = (
    df_daily
    .set_index('LogDate')
    .reindex(full_date_range)
    .rename_axis('LogDate')
    .reset_index()
)

# Fill missing days with 0 (no keys issued = system downtime or no access)
df_daily['Total_Daily_Keys'] = df_daily['Total_Daily_Keys'].fillna(0).astype(int)

print(f'After fill: {len(df_daily)} continuous days')
print(f'Null values: {df_daily["Total_Daily_Keys"].isnull().sum()}')

After fill: 398 continuous days
Null values: 0


In [10]:
# Outlier detection using IQR method
Q1 = df_daily['Total_Daily_Keys'].quantile(0.25)
Q3 = df_daily['Total_Daily_Keys'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = df_daily[
    (df_daily['Total_Daily_Keys'] < lower_bound) |
    (df_daily['Total_Daily_Keys'] > upper_bound)
]

print(f'IQR bounds: [{lower_bound:.0f}, {upper_bound:.0f}]')
print(f'Outlier days: {len(outliers)} ({len(outliers)/len(df_daily)*100:.1f}%)')
if len(outliers) > 0:
    display(outliers)

IQR bounds: [-5713, 9736]
Outlier days: 54 (13.6%)


,LogDate,Total_Daily_Keys
338,2020-04-03,14770
339,2020-04-04,15326
340,2020-04-05,15478
341,2020-04-06,13898
342,2020-04-07,13847
343,2020-04-08,13671
344,2020-04-09,13429
345,2020-04-10,13232
346,2020-04-11,13974
347,2020-04-12,14272


## Feature Engineering


In [11]:
df_daily['DayOfWeek'] = df_daily['LogDate'].dt.dayofweek          # 0=Mon, 6=Sun
df_daily['DayOfWeekName'] = df_daily['LogDate'].dt.day_name()
df_daily['IsWeekend'] = (df_daily['DayOfWeek'] >= 5).astype(int)
df_daily['DayOfMonth'] = df_daily['LogDate'].dt.day
df_daily['WeekOfYear'] = df_daily['LogDate'].dt.isocalendar().week.astype(int)
df_daily['Month'] = df_daily['LogDate'].dt.month
df_daily['Year'] = df_daily['LogDate'].dt.year
df_daily['Quarter'] = df_daily['LogDate'].dt.quarter

df_daily['IsMonthStart'] = df_daily['LogDate'].dt.is_month_start.astype(int)
df_daily['IsMonthEnd'] = df_daily['LogDate'].dt.is_month_end.astype(int)

print('Calendar features added.')
df_daily.head(3)

Calendar features added.


,LogDate,Total_Daily_Keys,DayOfWeek,DayOfWeekName,IsWeekend,DayOfMonth,WeekOfYear,Month,Year,Quarter,IsMonthStart,IsMonthEnd
0,2019-05-01,4589,2,Wednesday,0,1,18,5,2019,2,1,0
1,2019-05-02,3823,3,Thursday,0,2,18,5,2019,2,0,0
2,2019-05-03,4147,4,Friday,0,3,18,5,2019,2,0,0


In [12]:
# Rolling statistics
df_daily['MA_7'] = df_daily['Total_Daily_Keys'].rolling(window=7, min_periods=1).mean()
df_daily['MA_14'] = df_daily['Total_Daily_Keys'].rolling(window=14, min_periods=1).mean()
df_daily['MA_30'] = df_daily['Total_Daily_Keys'].rolling(window=30, min_periods=1).mean()
df_daily['Std_7'] = df_daily['Total_Daily_Keys'].rolling(window=7, min_periods=1).std()

# Lag features
df_daily['Lag_1'] = df_daily['Total_Daily_Keys'].shift(1)
df_daily['Lag_7'] = df_daily['Total_Daily_Keys'].shift(7)
df_daily['Lag_14'] = df_daily['Total_Daily_Keys'].shift(14)
df_daily['Lag_30'] = df_daily['Total_Daily_Keys'].shift(30)

# Week-over-week change
df_daily['WoW_Change'] = df_daily['Total_Daily_Keys'] - df_daily['Lag_7']
df_daily['WoW_Change_Pct'] = (
    df_daily['WoW_Change'] / df_daily['Lag_7'].replace(0, np.nan) * 100
)

print(f'Columns: {list(df_daily.columns)}')

Columns: ['LogDate', 'Total_Daily_Keys', 'DayOfWeek', 'DayOfWeekName', 'IsWeekend', 'DayOfMonth', 'WeekOfYear', 'Month', 'Year', 'Quarter', 'IsMonthStart', 'IsMonthEnd', 'MA_7', 'MA_14', 'MA_30', 'Std_7', 'Lag_1', 'Lag_7', 'Lag_14', 'Lag_30', 'WoW_Change', 'WoW_Change_Pct']


In [13]:
OUTPUT_DIR = os.path.join('..', 'Dataset')
os.makedirs(OUTPUT_DIR, exist_ok=True)

df_daily.to_csv(os.path.join(OUTPUT_DIR, 'daily_drm_keys_clean.csv'), index=False)
df_daily_source.to_csv(os.path.join(OUTPUT_DIR, 'daily_drm_keys_by_source.csv'), index=False)

print('Exported:')
print(f'  → {OUTPUT_DIR}/daily_drm_keys_clean.csv      ({len(df_daily)} rows)')
print(f'  → {OUTPUT_DIR}/daily_drm_keys_by_source.csv  ({len(df_daily_source)} rows)')

Exported:
  → ..\Dataset/daily_drm_keys_clean.csv      (398 rows)
  → ..\Dataset/daily_drm_keys_by_source.csv  (398 rows)


In [14]:
display(df_daily.tail(10))

,LogDate,Total_Daily_Keys,DayOfWeek,DayOfWeekName,IsWeekend,DayOfMonth,WeekOfYear,Month,Year,Quarter,IsMonthStart,IsMonthEnd,MA_7,MA_14,MA_30,Std_7,Lag_1,Lag_7,Lag_14,Lag_30,WoW_Change,WoW_Change_Pct
388,2020-05-23,13196,5,Saturday,1,23,21,5,2020,2,0,0,10715.00,10598.21,10751.03,1562.23,9541.00,12937.00,10343.00,12098.00,259.00,2.00
389,2020-05-24,12895,6,Sunday,1,24,21,5,2020,2,0,0,10747.14,10752.43,10770.60,1610.72,13196.00,12670.00,10736.00,12308.00,225.00,1.78
390,2020-05-25,10708,0,Monday,0,25,22,5,2020,2,0,0,10780.86,10815.29,10698.97,1606.47,12895.00,10472.00,9828.00,12857.00,236.00,2.25
391,2020-05-26,10207,1,Tuesday,0,26,22,5,2020,2,0,0,10819.71,10864.57,10604.07,1585.76,10708.00,9935.00,9517.00,13054.00,272.00,2.74
392,2020-05-27,10187,2,Wednesday,0,27,22,5,2020,2,0,0,10870.14,10872.50,10557.20,1554.49,10207.00,9834.00,10076.00,11593.00,353.00,3.59
393,2020-05-28,9856,3,Thursday,0,28,22,5,2020,2,0,0,10941.43,10849.93,10510.00,1483.35,10187.00,9357.00,10172.00,11272.00,499.00,5.33
394,2020-05-29,9842,4,Friday,0,29,22,5,2020,2,0,0,10984.43,10831.21,10464.00,1439.71,9856.00,9541.00,10104.00,11222.00,301.00,3.15
395,2020-05-30,12183,5,Saturday,1,30,22,5,2020,2,0,0,10839.71,10777.36,10498.93,1213.50,9842.00,13196.00,12937.00,11135.00,-1013.00,-7.68
396,2020-05-31,11883,6,Sunday,1,31,22,5,2020,2,0,1,10695.14,10721.14,10569.80,962.06,12183.00,12895.00,12670.00,9757.00,-1012.00,-7.85
397,2020-06-01,9089,0,Monday,0,1,23,6,2020,2,1,0,10463.86,10622.36,10533.27,1137.14,11883.00,10708.00,10472.00,10185.00,-1619.00,-15.12
